# 01 — Preparação dos dados

Baixa o **PlantVillage** (versão original colorida, 38 classes) e o **PlantDoc** (fotos de campo),
redimensiona tudo para **128×128** e salva um cache em `.npy` no Google Drive (`MyDrive/leafscan/cache`).
Os notebooks seguintes só leem esse cache.

**O que este notebook gera no Drive**
- `cache/pv_x.npy`, `pv_y.npy`, `pv_split.npy` — PlantVillage (split 0=treino, 1=validação, 2=teste; 70/15/15 estratificado)
- `cache/pd_x.npy`, `pd_y.npy`, `pd_split.npy` — PlantDoc (rótulos já no índice do PlantVillage; split 0=treino, 2=teste, oficial)
- `cache/meta.json` — lista de classes, mapeamento PlantDoc→PlantVillage e contagens
- `results/figs/pv_vs_plantdoc.png` — figura comparando as duas bases (para o artigo)

Tempo esperado: ~10–15 min. Pode rodar em GPU ou CPU (só usa CPU).

## 1. Configuração

In [3]:
from google.colab import drive
drive.mount('/content/drive')

import os, json, time
import numpy as np

DRIVE_DIR = '/content/drive/MyDrive/leafscan'
CACHE_DIR = f'{DRIVE_DIR}/cache'
FIG_DIR   = f'{DRIVE_DIR}/results/figs'
DATA_DIR  = '/content/datasets'
IMG_SIZE  = 128
SEED      = 42

for d in (CACHE_DIR, FIG_DIR, DATA_DIR):
    os.makedirs(d, exist_ok=True)
print('CPUs:', os.cpu_count())

Mounted at /content/drive
CPUs: 2


## 2. Download das bases
- PlantVillage: `spMohanty/PlantVillage-Dataset`, só a pasta `raw/color` (sparse checkout, ~54 mil imagens).
- PlantDoc: `pratikkayal/PlantDoc-Dataset` (pastas `train/` e `test/`).

In [4]:
%%bash
set -e
cd /content/datasets
if [ ! -d pv/raw/color ]; then
  git clone -q --depth 1 --filter=blob:none --sparse https://github.com/spMohanty/PlantVillage-Dataset pv
  cd pv && git sparse-checkout set raw/color && cd ..
fi
if [ ! -d plantdoc/train ]; then
  git clone -q --depth 1 https://github.com/pratikkayal/PlantDoc-Dataset plantdoc
fi
echo "PlantVillage: $(ls pv/raw/color | wc -l) classes, $(find pv/raw/color -type f | wc -l) arquivos"
echo "PlantDoc train: $(ls plantdoc/train | wc -l) classes, $(find plantdoc/train -type f | wc -l) arquivos"
echo "PlantDoc test:  $(ls plantdoc/test | wc -l) classes, $(find plantdoc/test -type f | wc -l) arquivos"

PlantVillage: 38 classes, 54305 arquivos
PlantDoc train: 28 classes, 2342 arquivos
PlantDoc test:  27 classes, 236 arquivos


## 3. Classes e mapeamento PlantDoc → PlantVillage
O PlantDoc tem 28 pastas; 27 têm equivalente direto no PlantVillage.
`Tomato two spotted spider mites leaf` fica de fora (só 2 imagens de treino e nenhuma de teste).

In [ ]:
PV_ROOT = f'{DATA_DIR}/pv/raw/color'
PD_ROOT = f'{DATA_DIR}/plantdoc'

PV_CLASSES = sorted(d for d in os.listdir(PV_ROOT) if os.path.isdir(os.path.join(PV_ROOT, d)))
assert len(PV_CLASSES) == 38, len(PV_CLASSES)
PV_INDEX = {c: i for i, c in enumerate(PV_CLASSES)}

PD_TO_PV = {
    'Apple Scab Leaf': 'Apple___Apple_scab',
    'Apple leaf': 'Apple___healthy',
    'Apple rust leaf': 'Apple___Cedar_apple_rust',
    'Bell_pepper leaf': 'Pepper,_bell___healthy',
    'Bell_pepper leaf spot': 'Pepper,_bell___Bacterial_spot',
    'Blueberry leaf': 'Blueberry___healthy',
    'Cherry leaf': 'Cherry_(including_sour)___healthy',
    'Corn Gray leaf spot': 'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot',
    'Corn leaf blight': 'Corn_(maize)___Northern_Leaf_Blight',
    'Corn rust leaf': 'Corn_(maize)___Common_rust_',
    'Peach leaf': 'Peach___healthy',
    'Potato leaf early blight': 'Potato___Early_blight',
    'Potato leaf late blight': 'Potato___Late_blight',
    'Raspberry leaf': 'Raspberry___healthy',
    'Soyabean leaf': 'Soybean___healthy',
    'Squash Powdery mildew leaf': 'Squash___Powdery_mildew',
    'Strawberry leaf': 'Strawberry___healthy',
    'Tomato Early blight leaf': 'Tomato___Early_blight',
    'Tomato Septoria leaf spot': 'Tomato___Septoria_leaf_spot',
    'Tomato leaf': 'Tomato___healthy',
    'Tomato leaf bacterial spot': 'Tomato___Bacterial_spot',
    'Tomato leaf late blight': 'Tomato___Late_blight',
    'Tomato leaf mosaic virus': 'Tomato___Tomato_mosaic_virus',
    'Tomato leaf yellow virus': 'Tomato___Tomato_Yellow_Leaf_Curl_Virus',
    'Tomato mold leaf': 'Tomato___Leaf_Mold',
    'grape leaf': 'Grape___healthy',
    'grape leaf black rot': 'Grape___Black_rot',
}
missing = [v for v in PD_TO_PV.values() if v not in PV_INDEX]
assert not missing, f'Classes do PlantVillage não encontradas: {missing}'
print(f'{len(PV_CLASSES)} classes PV, {len(PD_TO_PV)} classes PlantDoc mapeadas')

## 4. Função de leitura
Mesmo pré-processamento que o app vai usar: corrige rotação EXIF, converte para RGB,
**recorta o quadrado central** e redimensiona para 128×128 (bilinear).
As imagens do PlantVillage já são quadradas (256×256), então para elas é só o redimensionamento.

In [ ]:
from PIL import Image, ImageOps
from multiprocessing import Pool
from tqdm.auto import tqdm

IMG_EXT = ('.jpg', '.jpeg', '.png', '.bmp', '.webp')

def load_image(path, size=IMG_SIZE):
    try:
        with Image.open(path) as im:
            im = ImageOps.exif_transpose(im).convert('RGB')
            w, h = im.size
            s = min(w, h)
            left, top = (w - s) // 2, (h - s) // 2
            im = im.crop((left, top, left + s, top + s)).resize((size, size), Image.BILINEAR)
            return np.asarray(im, dtype=np.uint8)
    except Exception as e:
        return None

def load_many(paths, desc):
    x = np.empty((len(paths), IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
    ok = np.zeros(len(paths), dtype=bool)
    with Pool(os.cpu_count()) as pool:
        for i, arr in enumerate(tqdm(pool.imap(load_image, paths, chunksize=64), total=len(paths), desc=desc)):
            if arr is not None:
                x[i] = arr
                ok[i] = True
    bad = [p for p, k in zip(paths, ok) if not k]
    return x[ok], ok, bad

## 5. PlantVillage → cache (split 70/15/15 estratificado)

In [ ]:
from sklearn.model_selection import train_test_split

pv_paths, pv_labels = [], []
for c in PV_CLASSES:
    d = os.path.join(PV_ROOT, c)
    for f in sorted(os.listdir(d)):
        if f.lower().endswith(IMG_EXT):
            pv_paths.append(os.path.join(d, f))
            pv_labels.append(PV_INDEX[c])
pv_labels = np.array(pv_labels, dtype=np.int16)
print('Imagens PlantVillage:', len(pv_paths))

t0 = time.time()
pv_x, ok, pv_bad = load_many(pv_paths, 'PlantVillage')
pv_y = pv_labels[ok]
print(f'Lidas: {len(pv_x)} | falhas: {len(pv_bad)} | {time.time()-t0:.0f}s')

idx = np.arange(len(pv_y))
idx_tr, idx_tmp = train_test_split(idx, test_size=0.30, stratify=pv_y, random_state=SEED)
idx_va, idx_te = train_test_split(idx_tmp, test_size=0.50, stratify=pv_y[idx_tmp], random_state=SEED)
pv_split = np.zeros(len(pv_y), dtype=np.int8)
pv_split[idx_va] = 1
pv_split[idx_te] = 2
print('treino/val/teste:', [(pv_split == s).sum() for s in (0, 1, 2)])

t0 = time.time()
np.save(f'{CACHE_DIR}/pv_x.npy', pv_x)
np.save(f'{CACHE_DIR}/pv_y.npy', pv_y)
np.save(f'{CACHE_DIR}/pv_split.npy', pv_split)
print(f'Salvo no Drive em {time.time()-t0:.0f}s ({pv_x.nbytes/1e9:.2f} GB)')

## 6. PlantDoc → cache (split oficial)

In [ ]:
pd_paths, pd_labels, pd_split_list, pd_names = [], [], [], []
skipped = {}
for split_name, split_id in (('train', 0), ('test', 2)):
    root = os.path.join(PD_ROOT, split_name)
    for c in sorted(os.listdir(root)):
        d = os.path.join(root, c)
        if not os.path.isdir(d):
            continue
        files = [f for f in sorted(os.listdir(d)) if f.lower().endswith(IMG_EXT)]
        if c not in PD_TO_PV:
            skipped[f'{split_name}/{c}'] = len(files)
            continue
        for f in files:
            pd_paths.append(os.path.join(d, f))
            pd_labels.append(PV_INDEX[PD_TO_PV[c]])
            pd_split_list.append(split_id)
            pd_names.append(c)
print('Pastas ignoradas:', skipped)

pd_x, ok, pd_bad = load_many(pd_paths, 'PlantDoc')
pd_y = np.array(pd_labels, dtype=np.int16)[ok]
pd_split = np.array(pd_split_list, dtype=np.int8)[ok]
pd_names = [n for n, k in zip(pd_names, ok) if k]
print(f'Lidas: {len(pd_x)} | falhas: {len(pd_bad)}')
for p in pd_bad[:10]:
    print('  falhou:', p)
print('treino/teste:', (pd_split == 0).sum(), (pd_split == 2).sum())

np.save(f'{CACHE_DIR}/pd_x.npy', pd_x)
np.save(f'{CACHE_DIR}/pd_y.npy', pd_y)
np.save(f'{CACHE_DIR}/pd_split.npy', pd_split)

## 7. Metadados

In [ ]:
from collections import Counter

meta = {
    'img_size': IMG_SIZE,
    'seed': SEED,
    'pv_classes': PV_CLASSES,
    'pd_to_pv': PD_TO_PV,
    'pd_pv_indices': sorted({PV_INDEX[v] for v in PD_TO_PV.values()}),
    'pv_counts': {s: int((pv_split == i).sum()) for s, i in (('train', 0), ('val', 1), ('test', 2))},
    'pd_counts': {'train': int((pd_split == 0).sum()), 'test': int((pd_split == 2).sum())},
    'pd_test_per_class': dict(Counter(n for n, s in zip(pd_names, pd_split) if s == 2)),
    'pd_skipped_folders': skipped,
    'failed_files': [os.path.relpath(p, DATA_DIR) for p in pv_bad + pd_bad],
}
with open(f'{CACHE_DIR}/meta.json', 'w') as f:
    json.dump(meta, f, indent=2, ensure_ascii=False)
print(json.dumps({k: meta[k] for k in ('pv_counts', 'pd_counts', 'pd_skipped_folders')}, indent=2))
print('Arquivos com falha:', len(meta['failed_files']))

## 8. Figura: PlantVillage × PlantDoc (mesmas classes)

In [ ]:
import matplotlib.pyplot as plt

show = ['Tomato___Late_blight', 'Potato___Early_blight', 'Corn_(maize)___Common_rust_',
        'Apple___Apple_scab', 'Grape___Black_rot', 'Squash___Powdery_mildew']
rng = np.random.default_rng(SEED)
fig, axes = plt.subplots(2, len(show), figsize=(2.2 * len(show), 4.8))
for j, c in enumerate(show):
    k = PV_INDEX[c]
    i_pv = rng.choice(np.where(pv_y == k)[0])
    i_pd = rng.choice(np.where(pd_y == k)[0])
    axes[0, j].imshow(pv_x[i_pv]); axes[1, j].imshow(pd_x[i_pd])
    axes[0, j].set_title(c.replace('___', '\n').replace('_', ' '), fontsize=8)
    for a in axes[:, j]:
        a.set_xticks([]); a.set_yticks([])
axes[0, 0].set_ylabel('PlantVillage'); axes[1, 0].set_ylabel('PlantDoc')
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/pv_vs_plantdoc.png', dpi=200, bbox_inches='tight')
plt.show()

## 9. Conferência final
Se as três linhas abaixo aparecerem sem erro, o cache está pronto e você pode seguir para o `02_treino.ipynb`.

In [ ]:
for name in ('pv_x', 'pv_y', 'pv_split', 'pd_x', 'pd_y', 'pd_split'):
    a = np.load(f'{CACHE_DIR}/{name}.npy', mmap_mode='r')
    print(f'{name:9s} {str(a.shape):22s} {a.dtype}')
assert len(np.load(f'{CACHE_DIR}/pv_y.npy')) == len(np.load(f'{CACHE_DIR}/pv_split.npy'))
assert len(np.load(f'{CACHE_DIR}/pd_y.npy')) == len(np.load(f'{CACHE_DIR}/pd_split.npy'))
print('OK')